# Notebook 1: Exploratory Data Analysis & Feature Engineering
## Road Segment Congestion Pattern Discovery with K-Means

### 1. Project Context & Unit of Analysis Definition
In urban traffic telemetry, understanding congestion patterns is essential for municipal traffic signal control, public transport routing, and infrastructure planning.

#### What is a "Road Segment" in this Dataset?
- **Dataset Nature:** The raw dataset comprises continuous 15-minute sensor recordings at an arterial road checkpoint, logging vehicle counts (`CarCount`, `BikeCount`, `BusCount`, `TruckCount`), aggregated totals, and a qualitative `Traffic Situation` label.
- **Unit of Analysis:** Because observations stem from an individual arterial monitoring location over time, our empirical unit of analysis is the **time-slot** (hour of day $\times$ day of week profile).
- **Congestion Pattern Definition:** Congestion patterns represent **recurring temporal operating states** of this road segment (e.g., morning commuter peak, nighttime low-flow freight, weekend afternoon leisure).
- **Data Leakage Defense:** The column `Traffic Situation` is strictly **held out** and excluded from clustering. It is preserved exclusively for post-hoc unsupervised validation in Notebook 3.


In [ ]:
# Global Configuration and Reproducibility Setup
import sys
import os
sys.path.append(os.path.abspath("../src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from preprocessing import load_and_inspect_raw_data, engineer_features, prepare_clustering_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("Environment initialized with RANDOM_STATE =", RANDOM_STATE)


### 2. Raw Data Inspection & Data Hygiene
We inspect missing values, column data types, distribution anomalies, and duplicate records.


In [ ]:
raw_csv_path = "../data/raw/Traffic.csv"
df_raw, inspect_info = load_and_inspect_raw_data(raw_csv_path)

print("Dataset Shape:", inspect_info["shape"])
print("\nData Types:\n", pd.Series(inspect_info["dtypes"]))
print("\nMissing Counts per Column:\n", pd.Series(inspect_info["missing_counts"]))
print("\nDuplicate Rows:", inspect_info["duplicate_rows"])

df_raw.head()


### 3. Exploratory Data Analysis
Let's analyze vehicle distributions across hours of the day to detect morning/evening rush hours and nighttime freight operations.


In [ ]:
# Parse decimal hour for initial visual inspection
df_raw_eda = df_raw.copy()
parsed_time = pd.to_datetime(df_raw_eda["Time"], format="%I:%M:%S %p", errors="coerce").dt.time
df_raw_eda["hour"] = parsed_time.apply(lambda t: t.hour if pd.notnull(t) else np.nan)

hourly_means = df_raw_eda.groupby("hour")[["CarCount", "BikeCount", "BusCount", "TruckCount"]].mean()

fig, ax = plt.subplots(figsize=(10, 5))
hourly_means.plot(kind="bar", stacked=True, ax=ax, colormap="viridis")
plt.title("Diurnal Vehicle Composition by Hour of Day", fontsize=14, fontweight="bold")
plt.xlabel("Hour of Day (0 to 23)")
plt.ylabel("Mean Vehicle Count per 15-Min Interval")
plt.legend(["Cars", "Bikes/Motorcycles", "Buses", "Trucks"])
plt.tight_layout()
plt.show()


### 4. Feature Engineering & Multi-Collinearity Prevention
#### Methodological Rationale:
1. **Cyclical Time Encoding:** 23:45 is only 15 minutes away from 00:00. Linear representation (0 to 24) introduces artificial distance. We map time into trigonometric cyclical components:
   $$\sin\left(\frac{2\pi \cdot \text{hour}}{24}\right), \quad \cos\left(\frac{2\pi \cdot \text{hour}}{24}\right)$$
2. **Vehicle Mix Ratios:** Absolute counts are collinear with `Total`. To characterize road usage without linear dependence, we compute vehicle proportions (`car_ratio`, `bike_ratio`, `bus_ratio`, `truck_ratio`).
3. **Outlier Mitigation:** K-Means minimizes Euclidean distances and is sensitive to extreme sensor glitches. We compute IQR thresholds and winsorize `total_volume`.
4. **Held-Out Label:** `Traffic Situation` is separated from the feature space.


In [ ]:
df_engineered = engineer_features(df_raw)
print("Engineered Columns:", df_engineered.columns.tolist())
df_engineered[["Time", "hour_float", "sin_hour", "cos_hour", "is_weekend", "total_volume_capped", "truck_ratio"]].head()


### 5. Feature Selection & Correlation Check
We confirm that no two features carry direct mathematical redundancy.


In [ ]:
X_scaled_df, feature_cols, held_out_labels, scaler = prepare_clustering_matrix(
    df_engineered, scaler_path="../data/processed/scaler.joblib"
)

plt.figure(figsize=(8, 6))
sns.heatmap(X_scaled_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Scaled Clustering Features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Persist processed artifacts
df_engineered.to_csv("../data/processed/traffic_clean.csv", index=False)
X_scaled_df.to_csv("../data/processed/X_scaled.csv", index=False)
print("Saved cleaned data to data/processed/traffic_clean.csv")
print("Saved scaled matrix to data/processed/X_scaled.csv")
